In [1]:
!pip install pygame

In [2]:
import pygame
import time

pygame.mixer.init()

pygame.mixer.music.load(r"C:\Users\hp\Downloads\alarm sound wav.wav")

pygame.mixer.music.play()

time.sleep(5)

pygame.mixer.music.stop()

C:\Users\hp\anaconda3\Lib\site-packages\pygame\pkgdata.py:25: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import resource_stream, resource_exists


pygame 2.6.1 (SDL 2.28.4, Python 3.13.9)
Hello from the pygame community. https://www.pygame.org/contribute.html


In [ ]:
import cv2
import numpy as np
import time
import pygame
from tensorflow.keras.models import load_model


# Load model
model = load_model("eye_movement_model.keras")

classes = ["Closed", "Open"]


# Alarm setup
pygame.mixer.init()

pygame.mixer.music.load(
    r"C:\Users\hp\Downloads\alarm sound wav.wav"
)

alarm_on = False
closed_start_time = None


# Face and eye detection
face_cascade = cv2.CascadeClassifier(
    cv2.data.haarcascades + 
    "haarcascade_frontalface_default.xml"
)

eye_cascade = cv2.CascadeClassifier(
    cv2.data.haarcascades +
    "haarcascade_eye.xml"
)


# Webcam
import time
cap = cv2.VideoCapture(0)
closed_start_time = None
last_print_time = time.time()
while True:

    ret, frame = cap.read()

    if not ret:
        break


    gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)

    faces = face_cascade.detectMultiScale(
        gray,
        1.3,
        5
    )


    eye_status = "Open"


    for (x, y, w, h) in faces:

        roi_gray = gray[y:y+h, x:x+w]
        roi_color = frame[y:y+h, x:x+w]


        eyes = eye_cascade.detectMultiScale(
            roi_gray
        )


        for (ex, ey, ew, eh) in eyes[:2]:

            eye = roi_color[
                ey:ey+eh,
                ex:ex+ew
            ]


            # Preprocessing
            eye = cv2.cvtColor(
                eye,
                cv2.COLOR_BGR2RGB
            )

            eye = cv2.resize(
                eye,
                (224,224)
            )

            eye = eye.astype(
                np.float32
            ) / 255.0

            eye = np.expand_dims(
                eye,
                axis=0
            )


            # Prediction
            pred = model.predict(
                eye,
                verbose=0
            )


            label = classes[
                np.argmax(pred)
            ]
            # Print only once every 20 seconds
        current_time = time.time()

        if current_time - last_print_time >= 20:
            print("Eye status:", label)
            last_print_time = current_time

            eye_status = label


            print("Eye status:", label)


            cv2.rectangle(
                roi_color,
                (ex,ey),
                (ex+ew,ey+eh),
                (0,255,0),
                2
            )


            cv2.putText(
                roi_color,
                label,
                (ex,ey-10),
                cv2.FONT_HERSHEY_SIMPLEX,
                0.7,
                (0,255,0),
                2
            )



    # -------- Alarm Logic --------

    if eye_status == "Closed":


        if closed_start_time is None:
            closed_start_time = time.time()


        closed_duration = (
            time.time()
            -
            closed_start_time
        )


        print(
            "Closed seconds:",
            int(closed_duration)
        )


        cv2.putText(
            frame,
            f"Closed: {int(closed_duration)} sec",
            (30,50),
            cv2.FONT_HERSHEY_SIMPLEX,
            1,
            (0,0,255),
            2
        )


        # Test with 5 first
        if closed_duration >= 2 and not alarm_on:

            print("ALARM STARTED")

            pygame.mixer.music.play(-1)

            alarm_on = True



    else:

        closed_start_time = None


        if alarm_on:

            print("ALARM STOPPED")

            pygame.mixer.music.stop()

            alarm_on = False



    cv2.imshow(
        "Eye Detection",
        frame
    )


    if cv2.waitKey(1) & 0xFF == ord("q"):
        break



cap.release()

cv2.destroyAllWindows()

pygame.mixer.music.stop()

C:\Users\hp\anaconda3\Lib\site-packages\pygame\pkgdata.py:25: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import resource_stream, resource_exists


pygame 2.6.1 (SDL 2.28.4, Python 3.13.9)
Hello from the pygame community. https://www.pygame.org/contribute.html
